# STGAN — CNN spaziale + LSTM temporale

Questo notebook **addestra un nuovo detector**. Mantiene LSTM 168 h, calendario,
loss e score STGAN; sostituisce i blocchi sul grafo con una CNN spaziale.
Input: tre variabili su una finestra geografica più maschera delle celle presenti.
La baseline resta in `feat/stgan-paper`; gli output CNN hanno una directory propria.

Eseguire sul server che contiene il manifest preparato. Le analisi di previsione
rimangono posthoc e usano gli stessi CSV SDE-Net. Nessuna soglia viene ottimizzata
sul test: il top-1% è il budget globale di ranking predefinito.

In [ ]:
from pathlib import Path
import os
import sys
import json
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.anomaly_detection.stgan import STGANCNNConfig, STGAN, build_spatial_grid
from scripts.run_pvgis_stgan import run_stgan

manifest_candidates = [
    ROOT / 'outputs/pvgis_stgan/prepared/manifest.csv',
    ROOT / 'outputs/pvgis_stgan/prepared/manifest_shard_0000.csv',
    ROOT / 'outputs/pvgis_stgan_cnn/prepared/manifest.csv',
]
MANIFEST = Path(os.environ.get('STGAN_MANIFEST', str(next(
    (p for p in manifest_candidates if p.is_file()), manifest_candidates[0])))).resolve()
OUT_ROOT = Path(os.environ.get('STGAN_CNN_OUT_DIR',
    str(ROOT / 'outputs/pvgis_stgan_cnn/reference'))).resolve()
BASELINE_SEED_DIR = Path(os.environ.get('STGAN_BASELINE_SEED_DIR',
    str(ROOT / 'outputs/pvgis_stgan/paper_reference/seed_20'))).resolve()
SEEDS = (20,)
SEED = SEEDS[0]
SEED_DIR = OUT_ROOT / f'seed_{SEED}'
DEVICE = os.environ.get('STGAN_DEVICE', 'cuda')
TOP_K_PERCENT = 1.0
CONFIG = STGANCNNConfig(
    epochs=6, batch_size=256, hidden_size=64, n_layers=2,
    cnn_channels=32, cnn_layers=2, patch_size=3,
    recent_steps=1, trend_steps=168, train_samples_per_epoch=0,
    grid_crs=os.environ.get('STGAN_GRID_CRS', 'EPSG:32632'),
)
RUN_TRAINING = True  # False: consultazione di un run già completato
print('Manifest:', MANIFEST)
print('Output CNN:', OUT_ROOT)
display(pd.Series(CONFIG.to_dict(), name='Configurazione'))

## Verifica della disposizione geografica

Il codice verifica una griglia regolare e assegna le celle in ordine geografico.
Non riordina la lista kNN come se fosse un'immagine. Per i blocchi completi
controlla anche se l'insieme dei punti coincide con quello dei vicini originali.
I conteggi 917/232 vanno confermati sul manifest reale.
Le celle assenti rimangono mascherate; la località centrale è sempre presente.

In [ ]:
manifest = pd.read_csv(MANIFEST, dtype={'location': str})
grid = build_spatial_grid(manifest.latitude, manifest.longitude,
    patch_size=CONFIG.patch_size, grid_crs=CONFIG.grid_crs,
    grid_spacing=CONFIG.grid_spacing, grid_tolerance=CONFIG.grid_tolerance)
print(json.dumps(grid.metadata, indent=2))
grid_locations = grid.location_frame(manifest.location.tolist())
display(grid_locations.groupby('complete_patch').agg(
    localita=('location', 'size'), celle_valide_min=('n_valid_cells', 'min'),
    celle_valide_max=('n_valid_cells', 'max')))

fig, ax = plt.subplots(figsize=(7, 6))
for complete, label, color in ((True, 'Finestra completa', 'tab:blue'),
                               (False, 'Finestra incompleta', 'tab:orange')):
    selected = grid_locations.complete_patch.eq(complete)
    ax.scatter(grid.column_indices[selected], grid.row_indices[selected],
               s=12, color=color, label=label)
ax.invert_yaxis()
ax.set(xlabel='Colonna della griglia', ylabel='Riga della griglia',
       title='Copertura delle finestre geografiche', aspect='equal')
ax.legend()
plt.show()

In [ ]:
counts = STGAN(n_features=3, hidden_size=CONFIG.hidden_size,
    n_layers=CONFIG.n_layers, cnn_channels=CONFIG.cnn_channels,
    cnn_layers=CONFIG.cnn_layers, patch_size=CONFIG.patch_size).parameter_counts()
parameters = pd.DataFrame({
    'baseline_GCN_GRU': {'generator': 63171, 'discriminator': 36769},
    'CNN_con_maschera': counts,
})
parameters['differenza_pct'] = 100 * (
    parameters.CNN_con_maschera / parameters.baseline_GCN_GRU - 1)
display(parameters)
print('Baseline: configurazione originale 3 variabili, hidden 64, 2 layer, 9 nodi.')

## Addestramento

La prima esecuzione avvia il training. Il protocollo completo visita tutte le
coppie località–timestamp: può richiedere molto tempo. Per una prova ridotta
impostare `train_samples_per_epoch` e usare un output distinto; lo scoring
resta sull'intero test. Un run completato con la stessa configurazione viene
letto senza ripetere il training. Una directory parziale non viene sovrascritta.

In [ ]:
import hashlib
existing_metadata = OUT_ROOT / 'run_metadata.json'
if existing_metadata.is_file():
    saved = json.loads(existing_metadata.read_text())
    if (saved['configuration']['model'] != CONFIG.to_dict()
        or saved['paper_top_k_percent'] != TOP_K_PERCENT
        or saved['seeds'] != list(SEEDS)
        or saved.get('source_manifest_sha256') != hashlib.sha256(MANIFEST.read_bytes()).hexdigest()):
        raise ValueError('Il run salvato ha una configurazione diversa: scegliere un altro OUT_ROOT.')
    print('Run completato già presente:', OUT_ROOT)
elif RUN_TRAINING:
    run_stgan(manifest_path=MANIFEST, out_dir=OUT_ROOT,
        config=CONFIG, device=DEVICE, seeds=SEEDS,
        paper_top_k_percent=TOP_K_PERCENT)
else:
    print('Training disabilitato; le celle successive richiedono un run completato.')

In [ ]:
saved_run = json.loads((SEED_DIR.parent / 'run_metadata.json').read_text(encoding='utf-8'))
metadata = json.loads((SEED_DIR / 'metadata.json').read_text())
history = pd.read_csv(SEED_DIR / 'training_history.csv')
boundary = pd.read_csv(SEED_DIR / 'boundary_summary.csv')
display(boundary)
display(pd.Series(metadata['backend']['performance'], name='Prestazioni misurate'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, label in zip(axes, ['generator_loss', 'discriminator_loss'],
                             ['Loss generatore', 'Loss discriminatore']):
    ax.plot(history.epoch, history[column], marker='o')
    ax.set(xlabel='Epoca', ylabel=label)
    ax.grid(alpha=.25)
fig.tight_layout()
figure_dir = SEED_DIR / 'figures'
figure_dir.mkdir(exist_ok=True)
fig.savefig(figure_dir / 'training_losses.png', dpi=160)
plt.show()

summary_text = 'BEGIN_STGAN_CNN_SUMMARY\n' + json.dumps({
    'configuration': saved_run['configuration']['model'],
    'parameters': metadata['backend']['parameter_counts'],
    'grid': metadata['backend']['grid'],
    'performance': metadata['backend']['performance'],
    'boundary_summary': boundary.to_dict(orient='records'),
    'decision_rule': 'global test top-K; prima del filtro di qualità posthoc',
    'top_k_percent': metadata['paper_top_k_percent'],
}, indent=2) + '\nEND_STGAN_CNN_SUMMARY\n'
print(summary_text)
(SEED_DIR / 'model_results_summary.txt').write_text(summary_text, encoding='utf-8')

## Confronto opzionale con la baseline

Confronta le decisioni salvate sulle stesse coppie località–timestamp, leggendo
una località alla volta. Riporta esplicitamente timestamp non abbinati.
Interni e bordi sono separati secondo la finestra CNN. Per interpretare gli
interni come pari contesto verificare prima l'audit di coincidenza con il kNN.

Si tratta di **accordo tra detector**, non di accuratezza rispetto a vere anomalie.
Questa cella usa i top-K esportati senza filtro di qualità o nuovo ranking.
Le tre analisi posthoc mantengono invece il loro protocollo clean top-1%.

In [ ]:
from physiq_pv.reporting.stgan_cnn_comparison import compare_stgan_exports
if (BASELINE_SEED_DIR / 'locations').is_dir():
    agreement, agreement_locations = compare_stgan_exports(
        SEED_DIR, BASELINE_SEED_DIR, out_dir=SEED_DIR / 'baseline_comparison')
    display(agreement)
else:
    print('Baseline non disponibile:', BASELINE_SEED_DIR)

## Configurazioni per esperimenti successivi

Questa cella prepara una tabella; **non avvia una grid search** e non seleziona
sul test. Definire prima un criterio di validazione, budget e seed comuni.
La dimensione K cambia anche il numero di parametri del discriminatore.
K=1 usa solo la località centrale. I confronti tra K richiedono gli stessi target.

In [ ]:
from itertools import product
search_rows = []
for layers, channels, hidden, size in product((1, 2, 3), (16, 32), (32, 64), (1, 3, 5)):
    candidate = replace(CONFIG, hidden_size=hidden, cnn_layers=layers, cnn_channels=channels, patch_size=size)
    model = STGAN(n_features=3, hidden_size=candidate.hidden_size,
        n_layers=candidate.n_layers, cnn_channels=channels, cnn_layers=layers, patch_size=size)
    search_rows.append({'cnn_layers': layers, 'cnn_channels': channels,
                       'patch_size': size, 'hidden_size': hidden, **model.parameter_counts()})
search_plan = pd.DataFrame(search_rows)
display(search_plan)
search_plan.to_csv(SEED_DIR / 'candidate_configurations.csv', index=False)

## Riuso dei notebook di analisi

Impostare le variabili prima di avviare Jupyter, oppure nelle celle di
configurazione dei tre notebook, usando directory CNN distinte:

- `STGAN_SEED_DIR`: directory `seed_20` di questo run;
- `STGAN_POSTHOC_ROOT`: `outputs/sde_stgan_cnn_quality_filtered`;
- `STGAN_INPUT_TARGET_OUT_DIR`: `outputs/stgan_cnn_input_target_cases_t1_t6`;
- `ANOMALY_SENSITIVITY_OUT_DIR`: `outputs/anomaly_threshold_sensitivity_cnn_t1_t6`.

Rieseguire `stgan_pointwise_posthoc_sdenet.ipynb`,
`stgan_input_target_cases_sdenet.ipynb` e
`anomaly_threshold_sensitivity_mtgflow_stgan.ipynb`.
Usare gli stessi dati di qualità e le stesse previsioni della baseline.
Questi notebook producono le analisi aggregate su tutte le località.